# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', 'Dataset')}: {getattr(metadata, 'description', '')}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'n/a')}")
print(f"License: {getattr(metadata, 'license', 'n/a')}")
print(f"Keywords: {', '.join(getattr(metadata, 'keywords', []))}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's list the available record sets and the fields in each. You can reference any field or record set by its `@id`, which is the unique identifier from the Croissant schema.

In [ ]:
print('Record Sets in the dataset:')

# Get record sets
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    record_sets = []
    # fallback: As recordSet may be empty, try to inspect dataset for available records
    # We'll use dataset.records() to find the default record set IDs
    # mlcroissant exposes a utility for available record sets
    from mlcroissant.dataset import _find_record_set_ids
    record_sets = _find_record_set_ids(dataset._metadata_json_ld)
    print(f"  Automatically found {len(record_sets)} record set(s):")
else:
    print(f"  Found {len(record_sets)} record set(s):")

for idx, rec in enumerate(record_sets):
    # rec may be dict or string @id
    rec_id = rec['@id'] if isinstance(rec, dict) and '@id' in rec else rec
    print(f"  {idx+1}. {rec_id}")

# List all fields in each record set
print('\nFields for each record set (by @id):')
for rec in record_sets:
    rec_id = rec['@id'] if isinstance(rec, dict) and '@id' in rec else rec
    try:
        rec_md = dataset._find_record_set(rec_id)
        fields = rec_md.get('field', [])
        if not isinstance(fields, list):
            fields = [fields]
        print(f"\nRecord Set @id: {rec_id}")
        for f in fields:
            field_id = f['@id'] if isinstance(f, dict) and '@id' in f else f
            print(f"    Field @id: {field_id}")
    except Exception as e:
        print(f" (Could not load fields for record set {rec_id}: {e})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For this notebook, we'll extract the primary protein data table, which is typically named with a descriptive `@id` (e.g., ends with `/protein_table` or similar).

In [ ]:
# This block discovers and loads each available record set into DataFrames
if not record_sets:
    # fallback
    from mlcroissant.dataset import _find_record_set_ids
    record_sets = _find_record_set_ids(dataset._metadata_json_ld)

dataframes = {}
for rec in record_sets:
    rec_id = rec['@id'] if isinstance(rec, dict) and '@id' in rec else rec
    records = list(dataset.records(record_set=rec_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded {len(df)} records from record set @id: {rec_id}")
    else:
        print(f"No records found for record set @id: {rec_id}")

# For downstream analysis, choose the main protein table
if dataframes:
    # Heuristic: use the largest DataFrame (most records)
    main_record_set_id = max(dataframes, key=lambda rid: dataframes[rid].shape[0])
    print(f"\nDefaulting to main record set @id: {main_record_set_id}")
    print("Columns:\n", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

We'll select common columns such as total peptide count or molecular weight for demonstration. Replace the field `@id` with your actual field of interest as discovered above.

In [ ]:
# Pick a numeric field for filtering (update '@id' as needed)
df = dataframes[main_record_set_id]

# Example heuristics for common column names
candidate_fields = [col for col in df.columns if ('peptide' in col.lower() or 'count' in col.lower() or 'mw' in col.lower() or 'molecular_weight' in col.lower() or 'abundance' in col.lower()) and pd.api.types.is_numeric_dtype(df[col])]

if candidate_fields:
    numeric_field = candidate_fields[0]  # take the first match
else:
    # fallback: use first numeric column
    numerics = df.select_dtypes(include=['number']).columns
    numeric_field = numerics[0] if len(numerics) else df.columns[0]

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
print(f"Using numeric field: {numeric_field} (Threshold: {threshold:.2f})\n")

# Filter
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (n={len(filtered_df)})")

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
)
print(f"\nNormalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field
group_field = None
for col in df.columns:
    if col != numeric_field and df[col].nunique() < min(8, len(df)//4):
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (showing top 5 groups):")
    display(grouped_df.head())
else:
    print("No small categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field], bins=30, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# Boxplot by group (if group_field exists)
if group_field:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
In this notebook, we've loaded the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells FAIR^2 dataset with `mlcroissant`, examined its structure using `@id` references, and explored key protein-level numeric fields. This methodology can be extended to other Croissant-compliant datasets for reproducible, transparent data science and FAIR analysis.